In [2]:
import argparse
import random
import torch
import numpy as np

from geotransformer.utils.data import registration_collate_fn_stack_mode
from geotransformer.utils.torch import to_cuda, release_cuda
from geotransformer.utils.open3d import make_open3d_point_cloud, get_color, draw_geometries
from geotransformer.utils.registration import compute_registration_error

from config import make_cfg
from model import create_model

import open3d as o3d

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
scale = 1.0
ratio = 1.0

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)

In [5]:
#WEIGHTS = "../../output/with_aug/snapshots/epoch-40.pth.tar"
#WEIGHTS = '../../output/scale_inv/snapshots/epoch-40.pth.tar'
WEIGHTS = "../../output/geotransformer.facesdownsampledfixed.stage4.gse.k3.max.oacl.stage2.sinkhorn/snapshots/epoch-40.pth.tar"
cfg = make_cfg()
model = create_model(cfg).cuda()
state_dict = torch.load(WEIGHTS)
model.load_state_dict(state_dict["model"])

<All keys matched successfully>

In [6]:
REF_NUM = 20
#REF_NUM = 25

SRC_FILE = f"../../data/faces/demo/src_{REF_NUM}.npy"
#SRC_FILE = "plank_scaled.npy"
REF_FILE = f"../../data/faces/demo/ref_{REF_NUM}.npy"
GT_FILE = f"../../data/faces/demo/gt_{REF_NUM}.npy"
MORPHED_FULL_FILE = f"../../data/faces/demo/morphed_full_{REF_NUM}.npy"

In [7]:
def load_data(scale=1.0, ratio = 1.0):
    src_points = np.load(SRC_FILE)

    # scaling
    src_points *= scale

    # nonuniform downsampling
    num_points = src_points.shape[0]
    indices = np.random.choice(num_points, int(ratio*num_points), replace=False)
    src_points = src_points[indices]
    

    ref_points = np.load(REF_FILE)
    morphed_full_points = np.load(MORPHED_FULL_FILE)
    src_feats = np.ones_like(src_points[:, :1])
    ref_feats = np.ones_like(ref_points[:, :1])

    data_dict = {
        "ref_points": ref_points.astype(np.float32),
        "src_points": src_points.astype(np.float32),
        "ref_feats": ref_feats.astype(np.float32),
        "src_feats": src_feats.astype(np.float32),
        "morphed_full": morphed_full_points.astype(np.float32),
        "gt_z": np.zeros((32, 100), dtype=np.float32) 
    }

    if GT_FILE is not None:
        transform = np.load(GT_FILE)
        data_dict["transform"] = transform.astype(np.float32)

    return data_dict

def open3d_webrtc_draw(geometries):
    o3d.visualization.draw(geometries)
   

In [8]:
data_dict = load_data(scale=scale, ratio=ratio)
data_dict.keys()

dict_keys(['ref_points', 'src_points', 'ref_feats', 'src_feats', 'morphed_full', 'gt_z', 'transform'])

In [9]:
# prepare data
neighbor_limits = [38, 36, 36, 38]  # default setting in 3DMatch
data_dict = registration_collate_fn_stack_mode(
    [data_dict], cfg.backbone.num_stages, cfg.backbone.init_voxel_size, cfg.backbone.init_radius, neighbor_limits
)
# get results
ref_length_orig = data_dict['lengths'][0][0].item()
orig_points = data_dict['points'][0] # [ref; src]
ref_points = orig_points[:ref_length_orig]
src_points = orig_points[ref_length_orig:]

# prediction
data_dict = to_cuda(data_dict)
output_dict = model(data_dict)
data_dict = release_cuda(data_dict)
output_dict = release_cuda(output_dict)

estimated_transform = output_dict["estimated_transform"]
transform = data_dict["transform"]
pred_scale = output_dict["pred_scale"]

In [10]:
# visualization
ref_pcd = make_open3d_point_cloud(ref_points)
ref_pcd.estimate_normals()
ref_pcd.paint_uniform_color(get_color("custom_blue"))
src_pcd = make_open3d_point_cloud(src_points)
src_pcd.estimate_normals()
src_pcd.paint_uniform_color(get_color("custom_yellow"))
o3d.visualization.draw_plotly([ref_pcd, src_pcd])

In [11]:
pred_morphed_data = output_dict["morphed_full"]

if torch.is_tensor(pred_morphed_data):
    pred_morphed_data = pred_morphed_data.detach().cpu().numpy()
if pred_morphed_data.ndim == 3:
    pred_morphed_data = pred_morphed_data.squeeze(0)

pred_morphed_pcd = o3d.geometry.PointCloud()
pred_morphed_pcd.points = o3d.utility.Vector3dVector(pred_morphed_data)
pred_morphed_pcd.estimate_normals()
pred_morphed_pcd.paint_uniform_color([0.0, 1.0, 0.0]) # Green = Predicted Model Output

recon_gt_data = output_dict["recon_gt_points"]
if torch.is_tensor(recon_gt_data):
    recon_gt_data = recon_gt_data.detach().cpu().numpy()
if recon_gt_data.ndim == 3:
    recon_gt_data = recon_gt_data.squeeze(0)

recon_gt_pcd = o3d.geometry.PointCloud()
recon_gt_pcd.points = o3d.utility.Vector3dVector(recon_gt_data)
recon_gt_pcd.estimate_normals()
recon_gt_pcd.paint_uniform_color([0.0, 0.0, 1.0]) # Blue = GT PCA Reconstruction

print("Visualizing: Predicted Morphed Shape (Green) vs Ground Truth PCA Reconstruction (Blue)")
o3d.visualization.draw_plotly([pred_morphed_pcd, recon_gt_pcd])


Visualizing: Predicted Morphed Shape (Green) vs Ground Truth PCA Reconstruction (Blue)


In [12]:
import numpy as np
import open3d as o3d
import copy

# 1. Prepare parameters based on the tutorial logic
# Get the center and diameter of your specific point cloud
pcd_center = recon_gt_pcd.get_center()
diameter = np.linalg.norm(
    np.asarray(recon_gt_pcd.get_max_bound()) - np.asarray(recon_gt_pcd.get_min_bound())
)

# The tutorial suggests radius = diameter * 100
radius = diameter * 100
num_samples = 3
view_samples = []

print(f"Generating {num_samples} samples from random camera views...")

for i in range(num_samples):
    # 2. Define a random camera view point
    # We create a random direction vector and scale it by the diameter
    random_vec = np.random.randn(3)
    random_vec /= np.linalg.norm(random_vec) 
    
    # Place camera at a distance from the center
    camera = pcd_center + (random_vec * diameter)
    
    # 3. Apply Hidden Point Removal (as seen in Open3D tutorial)
    # This approximates visibility from the 'camera' viewpoint
    _, pt_map = recon_gt_pcd.hidden_point_removal(camera, radius)
    
    # 4. Select the visible points
    sample_pcd = recon_gt_pcd.select_by_index(pt_map)
    
    # Paint it a unique color so we can tell them apart
    color = np.random.rand(3)
    sample_pcd.paint_uniform_color(color)
    
    view_samples.append(sample_pcd)
    print(f"Sample {i+1} generated from camera: {camera}")

# 5. Visualize the results using Plotly (Notebook compatible)
# We show them side-by-side by translating them along the X axis
final_visuals = []
spacing = diameter * 1.2

# Assuming 'view_samples' contains the 3 point clouds generated in the previous step

for i, sample in enumerate(view_samples):
    print(f"Rendering Sample {i+1} (View Point {i+1})...")
    
    # We pass the individual sample as a list to draw_plotly
    o3d.visualization.draw_plotly([sample])

print("\nVisualizing the 3 samples side-by-side:")
o3d.visualization.draw_plotly(final_visuals)

Generating 3 samples from random camera views...
Sample 1 generated from camera: [ 1.93687138 -2.4213147  -1.80714167]
Sample 2 generated from camera: [1.54096609 1.27361725 3.13120744]
Sample 3 generated from camera: [3.46670105 0.94188346 0.88511964]
Rendering Sample 1 (View Point 1)...


Rendering Sample 2 (View Point 2)...


Rendering Sample 3 (View Point 3)...



Visualizing the 3 samples side-by-side:


/usr/local/lib/python3.12/dist-packages/open3d/visualization/draw_plotly.py:134: RuntimeWarning:

invalid value encountered in divide



In [13]:
# %%
results = []

# We'll use the original ref_points from your first load_data call as the static target
# Ensure ref_points and ref_feats are ready
ref_points_np = np.asarray(ref_pcd.points).astype(np.float32)
ref_feats_np = np.ones((ref_points_np.shape[0], 1), dtype=np.float32)

print(f"Starting forward passes for {len(view_samples)} random views...")

for i, view_pcd in enumerate(view_samples):
    # 1. Prepare data for this specific view
    view_points_np = np.asarray(view_pcd.points).astype(np.float32)
    view_feats_np = np.ones((view_points_np.shape[0], 1), dtype=np.float32)

    # Reconstruct a data_dict similar to what load_data() produces
    # Note: We use the same ref as the original for consistency
    sample_data_dict = {
        "ref_points": ref_points_np,
        "src_points": view_points_np,
        "ref_feats": ref_feats_np,
        "src_feats": view_feats_np,
        "gt_z": np.zeros((32, 100), dtype=np.float32), # Default placeholder
        "transform": np.eye(4).astype(np.float32)      # Identity since these are partial views of the GT
    }

    # 2. Collate and Preprocess (matching your neighbor_limits)
    neighbor_limits = [38, 36, 36, 38]
    collated_dict = registration_collate_fn_stack_mode(
        [sample_data_dict], cfg.backbone.num_stages, cfg.backbone.init_voxel_size, cfg.backbone.init_radius, neighbor_limits
    )

    # 3. Model Inference
    collated_dict = to_cuda(collated_dict)
    with torch.no_grad():
        view_output = model(collated_dict)
    
    collated_dict = release_cuda(collated_dict)
    view_output = release_cuda(view_output)

    # 4. Extract results
    est_transform = view_output["estimated_transform"]
    est_scale = view_output["pred_scale"]
    
    results.append({
        "pcd": view_pcd,
        "transform": est_transform,
        "scale": est_scale
    })
    
    print(f"View {i+1} processed. Predicted Scale: {est_scale:.4f}")

# %% [markdown]
# ### Visualize Predicted Transformations and Calculate Errors (RRE/RTE)

# %%
for i, res in enumerate(results):
    print(f"\n--- Analysis for View {i+1} ---")
    
    # 1. Compute Errors
    # 'transform' is the ground truth loaded from your GT_FILE earlier
    rre, rte = compute_registration_error(transform, res["transform"])
    
    # Calculate scale error (difference between loaded 'scale' and 'pred_scale')
    # Using the global 'scale' variable from your script's configuration
    scale_err = scale - res["scale"]
    
    print(f"RRE (deg): {rre:.3f}")
    print(f"RTE (m):   {rte:.3f}")
    print(f"Scale Error (GT - Pred): {scale_err:.3f}")
    print(f"Predicted Scale: {res['scale']:.4f}")

    # 2. Visualization Setup
    # Original Reference (Blue)
    ref_viz = copy.deepcopy(ref_pcd)
    ref_viz.paint_uniform_color([0.0, 0.0, 1.0]) # Blue
    
    # Source View (Transformed by model)
    # Logic: scale to unit space first, then apply estimated transform
    transformed_src = copy.deepcopy(res["pcd"])
    transformed_src.scale(1.0 / res["scale"], center=(0, 0, 0))
    transformed_src.transform(res["transform"])
    transformed_src.paint_uniform_color([1.0, 0.7, 0.0]) # Orange/Yellow
    
    # 3. Render
    print(f"Visualizing Alignment: Orange (Aligned Partial View) vs Blue (Reference)")
    o3d.visualization.draw_plotly([ref_viz, transformed_src])

Starting forward passes for 3 random views...
View 1 processed. Predicted Scale: 0.9639
View 2 processed. Predicted Scale: 1.0419
View 3 processed. Predicted Scale: 0.9945

--- Analysis for View 1 ---
RRE (deg): 80.377
RTE (m):   0.828
Scale Error (GT - Pred): 0.036
Predicted Scale: 0.9639
Visualizing Alignment: Orange (Aligned Partial View) vs Blue (Reference)



--- Analysis for View 2 ---
RRE (deg): 80.107
RTE (m):   0.860
Scale Error (GT - Pred): -0.042
Predicted Scale: 1.0419
Visualizing Alignment: Orange (Aligned Partial View) vs Blue (Reference)



--- Analysis for View 3 ---
RRE (deg): 80.097
RTE (m):   0.859
Scale Error (GT - Pred): 0.006
Predicted Scale: 0.9945
Visualizing Alignment: Orange (Aligned Partial View) vs Blue (Reference)


In [14]:
# %% [markdown]
# ### Superpoint Correspondence Visualization for 3 Random Views

# %%
from geotransformer.modules.ops import point_to_node_partition, index_select

# We'll use the original ref_pcd and its features as our constant target
ref_points_np = np.asarray(ref_pcd.points).astype(np.float32)
ref_feats_np = np.ones((ref_points_np.shape[0], 1), dtype=np.float32)

for i, view_pcd in enumerate(view_samples):
    print(f"\n--- Superpoint Matching for View {i+1} ---")
    
    # 1. Prepare data dict for this view
    view_points_np = np.asarray(view_pcd.points).astype(np.float32)
    view_feats_np = np.ones((view_points_np.shape[0], 1), dtype=np.float32)
    
    sample_data_dict = {
        "ref_points": ref_points_np,
        "src_points": view_points_np,
        "ref_feats": ref_feats_np,
        "src_feats": view_feats_np,
        "gt_z": np.zeros((32, 100), dtype=np.float32),
        "transform": np.eye(4).astype(np.float32) 
    }
    
    # 2. Collate for the model
    neighbor_limits = [38, 36, 36, 38]
    collated_dict = registration_collate_fn_stack_mode(
        [sample_data_dict], cfg.backbone.num_stages, cfg.backbone.init_voxel_size, cfg.backbone.init_radius, neighbor_limits
    )
    collated_dict = to_cuda(collated_dict)

    # 3. Manually extract coarse features (Superpoints)
    with torch.no_grad():
        # Get Backbone features
        feats_list = model.backbone(collated_dict['features'], collated_dict)
        feats_c = feats_list[-1] # Coarse features
        
        # Get point coordinates at different levels
        # Lengths tell us where reference ends and source begins
        ref_len_c = collated_dict['lengths'][-1][0].item()
        ref_len_f = collated_dict['lengths'][1][0].item()
        
        points_c = collated_dict['points'][-1] # Coarse points
        points_f = collated_dict['points'][1]  # Fine points
        
        ref_pts_c = points_c[:ref_len_c]
        src_pts_c = points_c[ref_len_c:]
        ref_pts_f = points_f[:ref_len_f]
        src_pts_f = points_f[ref_len_f:]
        
        # 4. Partition points to nodes (Internal Geotransformer logic)
        _, ref_node_masks, _, _ = point_to_node_partition(ref_pts_f, ref_pts_c, model.num_points_in_patch)
        _, src_node_masks, _, _ = point_to_node_partition(src_pts_f, src_pts_c, model.num_points_in_patch)

        # 5. Transformer & Coarse Matching
        ref_feats_c = feats_c[:ref_len_c]
        src_feats_c = feats_c[ref_len_c:]
        
        ref_feats_c, src_feats_c = model.transformer(
            ref_pts_c.unsqueeze(0), src_pts_c.unsqueeze(0),
            ref_feats_c.unsqueeze(0), src_feats_c.unsqueeze(0),
        )
        
        ref_feats_c_norm = torch.nn.functional.normalize(ref_feats_c.squeeze(0), p=2, dim=1)
        src_feats_c_norm = torch.nn.functional.normalize(src_feats_c.squeeze(0), p=2, dim=1)
        
        ref_corr_indices, src_corr_indices, _ = model.coarse_matching(
            ref_feats_c_norm, src_feats_c_norm, ref_node_masks, src_node_masks
        )

    # 6. Open3D Visualization Setup
    ref_pts_c_np = ref_pts_c.cpu().numpy()
    src_pts_c_np = src_pts_c.cpu().numpy()
    
    # Create Superpoint PCDs
    ref_nodes_pcd = o3d.geometry.PointCloud()
    ref_nodes_pcd.points = o3d.utility.Vector3dVector(ref_pts_c_np)
    ref_nodes_pcd.paint_uniform_color([1, 0, 0]) # Red Nodes
    
    src_nodes_pcd = o3d.geometry.PointCloud()
    src_nodes_pcd.points = o3d.utility.Vector3dVector(src_pts_c_np)
    src_nodes_pcd.paint_uniform_color([0, 1, 0]) # Green Nodes
    
    # 7. Create LineSet for Correspondences
    # Combine points for the LineSet indexing
    combined_pts = np.concatenate([ref_pts_c_np, src_pts_c_np], axis=0)
    lines = []
    for j in range(len(ref_corr_indices)):
        idx_ref = ref_corr_indices[j].item()
        idx_src = src_corr_indices[j].item() + ref_len_c # Offset by ref length
        lines.append([idx_ref, idx_src])
    
    line_set = o3d.geometry.LineSet()
    line_set.points = o3d.utility.Vector3dVector(combined_pts)
    line_set.lines = o3d.utility.Vector2iVector(lines)
    line_set.paint_uniform_color([0, 0, 1]) # Blue lines
    
    print(f"Showing {len(lines)} superpoint matches for View {i+1}")
    print("Red: Ref Superpoints | Green: Source Superpoints | Blue: Matches")
    o3d.visualization.draw_plotly([ref_nodes_pcd, src_nodes_pcd, line_set])


--- Superpoint Matching for View 1 ---
Showing 256 superpoint matches for View 1
Red: Ref Superpoints | Green: Source Superpoints | Blue: Matches



--- Superpoint Matching for View 2 ---
Showing 256 superpoint matches for View 2
Red: Ref Superpoints | Green: Source Superpoints | Blue: Matches



--- Superpoint Matching for View 3 ---
Showing 256 superpoint matches for View 3
Red: Ref Superpoints | Green: Source Superpoints | Blue: Matches
